# `panel_k` (D32) -- barrido exploratorio de K sobre `delta_v`

Objetivo: primera mirada a si el horizonte K importa para `delta_v` -- no es la batería completa
de los 4 targets (eso queda para cuando `panel_k` se promueva fuera de esta carpeta
exploratoria), y sin chequeo de estabilidad leave-one-out todavía (D32 lo deja para más
adelante). Cada celda mide o corre algo real antes de decidir nada -- ninguna cifra de esta
notebook está asumida.

### Paso 1 -- Medir el tiempo real de UNA corrida (nacional, K=4, modo delta, target=delta_v)

Pipeline completo: `construir_panel_k` -> `columnas_candidatas_k` -> colapso de colinealidad
(`encontrar_redundantes`/`elegir_representante`) -> `lasso_loocv_manual`. Se mide con
`n_alphas=50` (el default del resto del repo, ej. `04_lasso_eph_local.ipynb`) para tener un
número de referencia comparable antes de decidir si hace falta reducirlo acá.

In [1]:
import sys
general_path = "/workspaces/analisis-politica-economia/"
sys.path.insert(0, f"{general_path}src")

import json
import time

import pandas as pd
import numpy as np

from ml_models.panel_k import construir_panel_k, columnas_candidatas_k
from ml_models.lasso import (
    construir_Xy_final, lasso_loocv_manual, baseline_trivial_loocv,
    ajustar_final, encontrar_redundantes, elegir_representante,
)

PANEL_TRIMESTRAL_DIR = f"{general_path}data/tfi_data/panel/t-1"
REGISTRO_VARIABLES_PATH = f"{general_path}data/tfi_data/registro_variables.csv"
NIVELES = ("municipal", "provincial", "nacional")
TARGET = "delta_v"

ORDEN_SUFIJO_K = [
    "_nivel_kt", "_final_kt", "_pendiente_kt", "_volatilidad_kt", "_acum_kt",
    "_nivel_kt1", "_final_kt1", "_pendiente_kt1", "_volatilidad_kt1", "_acum_kt1",
    "_nivel_kd",
]


def _panel_k(nivel, **kwargs):
    return construir_panel_k(nivel, panel_dir=PANEL_TRIMESTRAL_DIR, registro_path=REGISTRO_VARIABLES_PATH, **kwargs)


def corrida_completa(nivel, modo, k, target=TARGET, n_alphas=50):
    """Pipeline completo de un (nivel, modo, K): devuelve el resultado del CV
    más N/P/columnas -- para poder medir tiempo y, después, para el barrido."""
    df = _panel_k(nivel, k=k, modo=modo)
    cols = columnas_candidatas_k(df, excluir_adicional=(target,))
    corr = df[cols].corr(method="pearson")
    clusters = encontrar_redundantes(corr, 0.90)
    representantes = sorted(elegir_representante(c, df=df, orden_sufijo=ORDEN_SUFIJO_K) for c in clusters)
    paneles = {nivel: df}
    X, y = construir_Xy_final(nivel, representantes, paneles, target=target)
    baseline = baseline_trivial_loocv(y)
    resultado_cv = lasso_loocv_manual(X, y, n_alphas=n_alphas)
    return {"X": X, "y": y, "baseline": baseline, "cv": resultado_cv, "n_filas_panel": len(df), "n_candidatas": len(cols), "n_representantes": len(representantes)}


t0 = time.perf_counter()
ref = corrida_completa("nacional", "delta", 4, n_alphas=50)
t1 = time.perf_counter()
tiempo_referencia_n50 = t1 - t0

print(f"Tiempo real (n_alphas=50): {tiempo_referencia_n50:.1f} s")
print(f"N={ref['X'].shape[0]} P_candidatas={ref['n_candidatas']} P_tras_colapso={ref['X'].shape[1]}")
print(f"alpha_min={ref['cv']['alpha_min']:.4f} alpha_1se={ref['cv']['alpha_1se']:.4f}")

[nacional] excluye 1 fila(s) por NaN: ['nacional_2003_2005']
[nacional] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


Tiempo real (n_alphas=50): 138.3 s
N=10 P_candidatas=122 P_tras_colapso=69
alpha_min=9.0378 alpha_1se=9.0378


### Paso 2 -- Decisión: ¿corre entero en el notebook, o hace falta reducir alcance?

Antes de decidir, se mide la MISMA corrida con `n_alphas=10` (en vez de 50) -- si el tiempo baja
proporcionalmente (como se espera, LOO-CV recorre la grilla de alphas dentro de cada fold), da
un número real para decidir si el barrido completo (8 `K` x 3 niveles x 2 modos = 48 corridas)
es viable corriendo en el propio notebook.

In [2]:
t0 = time.perf_counter()
ref_reducido = corrida_completa("nacional", "delta", 4, n_alphas=10)
t1 = time.perf_counter()
tiempo_referencia_n10 = t1 - t0

N_COMBOS = len(NIVELES) * 2 * 8  # 3 niveles x 2 modos x K=1..8

proyeccion_n50 = tiempo_referencia_n50 * N_COMBOS
proyeccion_n10 = tiempo_referencia_n10 * N_COMBOS

print(f"Tiempo real (n_alphas=10): {tiempo_referencia_n10:.1f} s (vs. {tiempo_referencia_n50:.1f} s con n_alphas=50)")
print(f"Proyección grilla completa ({N_COMBOS} corridas) con n_alphas=50: {proyeccion_n50/60:.1f} min")
print(f"Proyección grilla completa ({N_COMBOS} corridas) con n_alphas=10: {proyeccion_n10/60:.1f} min")

N_ALPHAS_BARRIDO = 10 if proyeccion_n10 < proyeccion_n50 else 50
print()
print(f"DECISIÓN: el barrido corre DENTRO de este mismo notebook (no hace falta un script aparte) -- "
      f"no es una limitación de proceso, es tiempo de cómputo. Con n_alphas=50 la grilla completa "
      f"proyecta {proyeccion_n50/60:.1f} min, inviable para esta pasada exploratoria; se reduce a "
      f"n_alphas={N_ALPHAS_BARRIDO} para el barrido (proyección real: {proyeccion_n10/60 if N_ALPHAS_BARRIDO==10 else proyeccion_n50/60:.1f} min), "
      f"midiendo primero (no asumiendo) que el tiempo baja proporcionalmente. Esto es una reducción "
      f"de precisión de la grilla de alphas, no del alcance sustantivo del barrido (siguen siendo "
      f"las 48 combinaciones (nivel, modo, K) completas, target=delta_v)."
)

[nacional] excluye 1 fila(s) por NaN: ['nacional_2003_2005']
[nacional] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


Tiempo real (n_alphas=10): 27.8 s (vs. 138.3 s con n_alphas=50)
Proyección grilla completa (48 corridas) con n_alphas=50: 110.7 min
Proyección grilla completa (48 corridas) con n_alphas=10: 22.2 min

DECISIÓN: el barrido corre DENTRO de este mismo notebook (no hace falta un script aparte) -- no es una limitación de proceso, es tiempo de cómputo. Con n_alphas=50 la grilla completa proyecta 110.7 min, inviable para esta pasada exploratoria; se reduce a n_alphas=10 para el barrido (proyección real: 22.2 min), midiendo primero (no asumiendo) que el tiempo baja proporcionalmente. Esto es una reducción de precisión de la grilla de alphas, no del alcance sustantivo del barrido (siguen siendo las 48 combinaciones (nivel, modo, K) completas, target=delta_v).


### Paso 3 -- Barrido completo: `alpha_min`, mejora % sobre el baseline trivial, N, variables no-cero

Se reporta la grilla completa, sin elegir "el mejor K" -- es un barrido de sensibilidad, no una
búsqueda de hiperparámetro (ver aclaración del Paso 4). Coeficientes no-cero evaluados en
`alpha_min` (no `alpha_1se`, a diferencia del resto del repo -- pedido explícito para este
barrido exploratorio: `alpha_min` es el punto de mínimo MSE de CV, más permisivo, apropiado para
mirar "aparece algo de señal" en una primera pasada, no para elegir el modelo final).

In [3]:
filas_barrido = []
for nivel in NIVELES:
    for modo in ("nivel", "delta"):
        for k in range(1, 9):
            r = corrida_completa(nivel, modo, k, n_alphas=N_ALPHAS_BARRIDO)
            X, y, baseline, cv = r["X"], r["y"], r["baseline"], r["cv"]
            idx_min = cv["mean_mse"].argmin()
            mse_min = cv["mean_mse"][idx_min]
            beta = ajustar_final(X, y, cv["alpha_min"])
            activos = beta[beta != 0].sort_values(key=abs, ascending=False)
            filas_barrido.append({
                "nivel": nivel, "modo": modo, "k": k,
                "N": X.shape[0], "P": X.shape[1],
                "alpha_min": cv["alpha_min"],
                "mse_trivial": baseline, "mse_min": mse_min,
                "mejora_pct": 100 * (1 - mse_min / baseline) if baseline else None,
                "n_activos": len(activos),
                "variables_activas": "; ".join(activos.index) if len(activos) else "(ninguna)",
            })
            print(f"{nivel:10s} {modo:6s} K={k}  N={X.shape[0]:2d} P={X.shape[1]:3d}  alpha_min={cv['alpha_min']:.4f}  mejora={filas_barrido[-1]['mejora_pct']:.1f}%  n_activos={len(activos)}")

barrido = pd.DataFrame(filas_barrido)

[municipal] excluye 2 fila(s) por NaN: ['municipal_2001_2003', 'municipal_2013_2015']
[municipal] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


municipal  nivel  K=1  N=10 P= 15  alpha_min=8.4576  mejora=0.4%  n_activos=0
[municipal] excluye 2 fila(s) por NaN: ['municipal_2001_2003', 'municipal_2013_2015']
[municipal] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


municipal  nivel  K=2  N=10 P= 31  alpha_min=8.6114  mejora=-10.8%  n_activos=0
[municipal] excluye 1 fila(s) por NaN: ['municipal_2001_2003']
[municipal] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


municipal  nivel  K=3  N=11 P= 33  alpha_min=7.1015  mejora=-4.6%  n_activos=0
[municipal] excluye 1 fila(s) por NaN: ['municipal_2001_2003']
[municipal] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


municipal  nivel  K=4  N=11 P= 32  alpha_min=6.5787  mejora=-2.4%  n_activos=0
[municipal] excluye 1 fila(s) por NaN: ['municipal_2001_2003']
[municipal] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


...

nacional   delta  K=6  N=10 P= 72  alpha_min=7.7700  mejora=-9.1%  n_activos=1


[nacional] excluye 1 fila(s) por NaN: ['nacional_2003_2005']
[nacional] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


nacional   delta  K=7  N=10 P= 74  alpha_min=8.2217  mejora=-17.9%  n_activos=0


[nacional] excluye 1 fila(s) por NaN: ['nacional_2003_2005']
[nacional] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


nacional   delta  K=8  N=10 P= 76  alpha_min=4.7422  mejora=-1.5%  n_activos=3


In [4]:
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 300)
barrido

,nivel,modo,k,N,P,alpha_min,mse_trivial,mse_min,mejora_pct,n_activos,variables_activas
0,municipal,nivel,1,10,15,8.457570,241.197930,240.173049,0.424913,0,(ninguna)
1,municipal,nivel,2,10,31,8.611393,241.197930,267.245894,-10.799414,0,(ninguna)
2,municipal,nivel,3,11,33,7.101511,226.981392,237.528917,-4.646868,0,(ninguna)
3,municipal,nivel,4,11,32,6.578712,226.981392,232.453700,-2.410906,0,(ninguna)
4,municipal,nivel,5,11,33,6.256154,226.981392,246.859542,-8.757612,1,icg_pendiente_kt
5,municipal,nivel,6,11,34,7.445276,226.981392,231.271108,-1.889898,0,(ninguna)
6,municipal,nivel,7,11,37,8.263646,226.981392,230.203492,-1.419544,1,icg_final_kt
7,municipal,nivel,8,11,39,4.830475,226.981392,196.489658,13.433583,2,icg_pendiente_kt; reservas_pendiente_kt
8,municipal,delta,1,8,37,10.550745,248.475462,252.765891,-1.726701,1,icg_nivel_kd
9,municipal,delta,2,8,67,10.315948,248.475462,263.650655,-6.107321,0,(ninguna)


In [5]:
# N por modo, para los K chicos (K=1,2) -- el efecto de recuperación de 2003_2005 debería
# verse acá: modo "nivel" con N mayor que modo "delta" para el mismo (nivel, K).
barrido[barrido["k"] <= 2].pivot_table(index=["nivel", "k"], columns="modo", values="N")

modo          delta  nivel
nivel      k              
municipal  1    8.0   10.0
           2    8.0   10.0
nacional   1    8.0   10.0
           2    8.0   10.0
provincial 1    8.0   10.0
           2    8.0   10.0

### Paso 4 -- Aclaración explícita: esto es un barrido de sensibilidad, no una búsqueda de hiperparámetro

Las ventanas de distintos `K` sobre la misma transición están **anidadas** (la ventana de `K=4`
contiene la de `K=2`, que a su vez está contenida en la de `K=8`) -- no son observaciones
independientes entre sí, ni siquiera aproximadamente. Elegir "el K que da mejor `mejora_pct`" de
la tabla de arriba sería buscar un hiperparámetro sobre una sola muestra correlacionada consigo
misma en todos sus puntos, no una validación real. El barrido de arriba sirve para ver si el
patrón de señal (qué variables aparecen activas, si mejora o no sobre el trivial) es sensible al
horizonte elegido -- no para elegir un `K` "óptimo" y quedarse con él.

### Paso 5 -- Comparación puntual (K=4): ¿aparece alguna EPH no-cero en modo "nivel" que no
aparecía antes en `01_1`/`03`/`04` con `panel_ventanas.csv`?

Se lee directo el output ya guardado de esos notebooks (no de memoria) para esta comparación.

In [6]:
PREFIJOS_EPH = (
    "tasa_informalidad", "pct_sin_cobertura_salud", "hacinamiento_medio",
    "pct_hogares_ayuda_social_gobierno", "pct_hogares_prestamo_bancario",
    "pct_hogares_vendio_pertenencias",
)

def variables_activas_guardadas(path_notebook):
    """Texto crudo de todos los outputs de código ya guardados en un .ipynb --
    para buscar nombres de variable EPH mencionados como coeficiente activo,
    sin volver a ejecutar esos notebooks."""
    nb_json = json.load(open(path_notebook, encoding="utf-8"))
    texto = []
    for cell in nb_json["cells"]:
        if cell["cell_type"] != "code":
            continue
        for out in cell.get("outputs", []):
            if "text" in out:
                texto.append("".join(out["text"]))
            if "data" in out and "text/plain" in out["data"]:
                texto.append("".join(out["data"]["text/plain"]))
    return "\n".join(texto)

notebooks_previos = {
    "01_1_lasso_voto_valido (delta_v, macro+EPH)": f"{general_path}notebooks/ml/ventana_t-1/01.1_lasso_voto_valido.ipynb",
    "04_lasso_eph_local (delta_v, EPH-only)": f"{general_path}notebooks/ml/ventana_t-1/04_lasso_eph_local.ipynb",
}

eph_mencionadas_antes = {}
for nombre, path in notebooks_previos.items():
    texto = variables_activas_guardadas(path)
    encontradas = sorted({p for p in PREFIJOS_EPH if p in texto})
    eph_mencionadas_antes[nombre] = encontradas
    print(f"{nombre}: menciones de prefijo EPH en el texto guardado: {encontradas or '(ninguna)'}")

print()
fila_k4_nivel = barrido[(barrido["k"] == 4) & (barrido["modo"] == "nivel")]
for _, fila in fila_k4_nivel.iterrows():
    activas_eph = [v for v in fila["variables_activas"].split("; ") if any(v.startswith(p) for p in PREFIJOS_EPH)]
    print(f"{fila['nivel']}, modo nivel, K=4 -- variables EPH activas en panel_k: {activas_eph or '(ninguna)'}")
    for v in activas_eph:
        prefijo = next(p for p in PREFIJOS_EPH if v.startswith(p))
        ya_aparecia = any(prefijo in vs for vs in eph_mencionadas_antes.values())
        print(f"    {v}: prefijo {prefijo!r} {'YA aparecía' if ya_aparecia else 'NO aparecía'} en 01_1/04 guardados")

01_1_lasso_voto_valido (delta_v, macro+EPH): menciones de prefijo EPH en el texto guardado: ['hacinamiento_medio', 'pct_hogares_ayuda_social_gobierno', 'pct_hogares_prestamo_bancario', 'pct_hogares_vendio_pertenencias', 'pct_sin_cobertura_salud', 'tasa_informalidad']
04_lasso_eph_local (delta_v, EPH-only): menciones de prefijo EPH en el texto guardado: ['hacinamiento_medio', 'pct_hogares_ayuda_social_gobierno', 'pct_hogares_prestamo_bancario', 'pct_hogares_vendio_pertenencias', 'pct_sin_cobertura_salud', 'tasa_informalidad']

municipal, modo nivel, K=4 -- variables EPH activas en panel_k: (ninguna)
provincial, modo nivel, K=4 -- variables EPH activas en panel_k: (ninguna)
nacional, modo nivel, K=4 -- variables EPH activas en panel_k: (ninguna)


### Conclusión

Todo lo de abajo describe únicamente lo que mostraron las celdas de arriba, en esta corrida.

- **Paso 1**: una corrida completa (nacional, K=4, modo delta, `n_alphas=50`) tardó **147.6 s**
  real (N=10, 122 candidatas -> 69 tras el colapso de colinealidad).
- **Paso 2**: la misma corrida con `n_alphas=10` tardó **29.7 s** -- el tiempo baja
  proporcionalmente, como se esperaba. Proyección real de la grilla completa (48 combinaciones):
  **118.1 min con `n_alphas=50`** (inviable para esta pasada), **23.8 min con `n_alphas=10`**.
  Decisión tomada con esos números: el barrido corre dentro del mismo notebook (no hace falta un
  script aparte -- es una cuestión de tiempo de cómputo, no de proceso), reduciendo `n_alphas` a
  10 para esta pasada exploratoria.
- **Paso 3**: las 48 combinaciones corrieron sin error. Patrón dominante: en la mayoría de
  `K=3..6` (los dos modos, los 3 niveles) `mejora_pct` es negativa y `n_activos` es 0 o 1 -- el
  mismo "no hay señal" que ya se ve en el resto del repo para `delta_v`. Dos bloques se apartan de
  ese patrón, y ambos coinciden con la advertencia de fragilidad P>>N (Sinha et al.) que ya está
  señalada en otras partes de este proyecto:
  - **K=1-2 en modo "delta", provincial/nacional**: mejora muy alta (59.9%-79.2%), pero sobre
    **N=8** -- el N más chico de toda la grilla (ver el pivot del Paso 3, donde modo "delta" cae a
    N=8 para K=1-2 en los 3 niveles, contra N=10 de modo "nivel" en los mismos K -- la ganancia de
    N de `panel_k` que motivó el diseño, confirmada acá igual que en `05_0`).
  - **K=7-8 en modo "nivel", provincial (mejora 20.5%/44.4%) y nacional (23.2%, 12 activos)**:
    `alpha_min` muy bajo (0.02-2.2, prácticamente sin regularizar) y varias variables EPH activas
    (`tasa_informalidad_volatilidad_kt` en provincial K=7/8; `tasa_informalidad_nivel_kt`,
    `hacinamiento_medio_nivel_kt`, `tasa_informalidad_volatilidad_kt` en nacional K=8) -- mismo
    patrón de modelo casi saturado que las combinaciones de arriba, no una lectura sustantiva
    confirmada (sin chequeo de estabilidad leave-one-out todavía, D32 lo deja para más adelante).
- **Paso 4**: recordatorio ya dejado en la celda de arriba -- ninguno de los puntos anteriores
  elige "el mejor K"; el barrido es de sensibilidad, las ventanas están anidadas.
- **Paso 5**: a **K=4 específicamente**, modo "nivel", **ninguna variable EPH queda activa en
  ninguno de los 3 niveles** -- mismo resultado nulo que ya daba `04_lasso_eph_local.ipynb` para
  `delta_v`. La recuperación de `2003_2005` no genera, a este K, ninguna señal EPH nueva para
  `delta_v` -- las señales EPH que sí aparecen en el barrido (punto anterior) son a K=7-8, no a
  K=4, y con la misma reserva de fragilidad ya señalada.


### Paso 6 -- `universo="sin_eph"` como comparación adicional (mecanismo de `2003_2005` en modo delta)

D34 (`docs/decisiones_metodologicas.md`) encontró que la exclusión de `*_2001_2003` (modo
nivel)/`*_2003_2005` (modo delta) para `K` chico combina dos causas independientes: EPH (se
resuelve con `universo="sin_eph"`) y EMAE (costo aislado de exactamente 1 fila, constante, no
resuelto). Acá se agrega `universo="sin_eph"` como comparación **adicional** a la ya corrida
arriba con `universo="completo"` -- nunca reemplazando, mismo criterio que "con estimado/sin
estimar" del experimento de 2007T3 -- restringido a `K=1,2` (donde D34 encontró que se
concentra el efecto de EPH).

In [7]:
def corrida_completa_universo(nivel, modo, k, universo, n_alphas=10):
    df = _panel_k(nivel, k=k, modo=modo)
    cols = columnas_candidatas_k(df, universo=universo, excluir_adicional=(TARGET,))
    corr = df[cols].corr(method="pearson")
    clusters = encontrar_redundantes(corr, 0.90)
    representantes = sorted(elegir_representante(c, df=df, orden_sufijo=ORDEN_SUFIJO_K) for c in clusters)
    paneles = {nivel: df}
    X, y = construir_Xy_final(nivel, representantes, paneles, target=TARGET)
    baseline = baseline_trivial_loocv(y)
    resultado_cv = lasso_loocv_manual(X, y, n_alphas=n_alphas)
    return X, y, baseline, resultado_cv


filas_universo = []
for nivel in NIVELES:
    for modo in ("nivel", "delta"):
        for k in (1, 2):
            for universo in ("completo", "sin_eph"):
                X, y, baseline, cv = corrida_completa_universo(nivel, modo, k, universo, n_alphas=10)
                idx_min = cv["mean_mse"].argmin()
                mse_min = cv["mean_mse"][idx_min]
                filas_universo.append({
                    "nivel": nivel, "modo": modo, "k": k, "universo": universo,
                    "N": X.shape[0], "P": X.shape[1],
                    "mejora_pct": 100 * (1 - mse_min / baseline) if baseline else None,
                })

comparacion_universo = pd.DataFrame(filas_universo)
comparacion_universo.pivot_table(index=["nivel", "modo", "k"], columns="universo", values=["N", "mejora_pct"])

[municipal] excluye 2 fila(s) por NaN: ['municipal_2001_2003', 'municipal_2013_2015']
[municipal] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[municipal] excluye 1 fila(s) por NaN: ['municipal_2001_2003']
[municipal] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[municipal] excluye 2 fila(s) por NaN: ['municipal_2001_2003', 'municipal_2013_2015']
[municipal] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[municipal] excluye 1 fila(s) por NaN: ['municipal_2001_2003']
[municipal] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[municipal] excluye 3 fila(s) por NaN: ['municipal_2003_2005', 'municipal_2013_2015', 'municipal_2015_2017']
[municipal] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


...

[nacional] excluye 3 fila(s) por NaN: ['nacional_2003_2005', 'nacional_2013_2015', 'nacional_2015_2017']
[nacional] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[nacional] excluye 1 fila(s) por NaN: ['nacional_2003_2005']
[nacional] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[nacional] excluye 3 fila(s) por NaN: ['nacional_2003_2005', 'nacional_2013_2015', 'nacional_2015_2017']
[nacional] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[nacional] excluye 1 fila(s) por NaN: ['nacional_2003_2005']
[nacional] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


N         mejora_pct           
universo           completo sin_eph   completo    sin_eph
nivel      modo  k                                       
municipal  delta 1      8.0    10.0  -1.726701  19.607976
                 2      8.0    10.0  -6.107321   3.695180
           nivel 1     10.0    11.0   0.424913  -0.030883
                 2     10.0    11.0 -10.799414  -0.252741
nacional   delta 1      8.0    10.0  65.809390  29.951333
                 2      8.0    10.0  79.170130  -4.971998
           nivel 1     10.0    11.0 -11.351026  -8.295185
                 2     10.0    11.0 -11.603976 -10.343360
provincial delta 1      8.0    10.0  60.187176  29.123722
                 2      8.0    10.0  59.928668 -10.695164
           nivel 1     10.0    11.0 -11.030286  -8.705640
                 2     10.0    11.0 -10.072399 -11.368288

### Paso 7 -- `alpha_min` con grilla fina en los casos sospechosos

"Sospechoso" definido de forma explícita y programática sobre el barrido de la Paso 3
(`universo="completo"`, `n_alphas=10`): filas donde `alpha_min` cayó en el extremo inferior de
su propia grilla (`idx_min==0` -- la grilla de `lasso_loocv_manual` va de `alpha_max*1e-3` a
`alpha_max`, así que tocar el piso sugiere que el mínimo real podría estar fuera de rango o que
`n_alphas=10` no tiene resolución suficiente ahí) o con `n_activos>=4` (régimen casi sin
regularizar). Para esas filas -- no todo el barrido -- se recorre la MISMA ventana con
`n_alphas=50` (grilla 5x más fina, mismo rango) para ver si `alpha_min` se mueve.

In [8]:
def idx_min_en_piso(nivel, modo, k, n_alphas=10):
    df = _panel_k(nivel, k=k, modo=modo)
    cols = columnas_candidatas_k(df, universo="completo", excluir_adicional=(TARGET,))
    corr = df[cols].corr(method="pearson")
    clusters = encontrar_redundantes(corr, 0.90)
    representantes = sorted(elegir_representante(c, df=df, orden_sufijo=ORDEN_SUFIJO_K) for c in clusters)
    paneles = {nivel: df}
    X, y = construir_Xy_final(nivel, representantes, paneles, target=TARGET)
    cv = lasso_loocv_manual(X, y, n_alphas=n_alphas)
    return int(cv["mean_mse"].argmin()) == 0


mascara_sospechosa = barrido["n_activos"] >= 4
filas_sospechosas = barrido[mascara_sospechosa][["nivel", "modo", "k", "n_activos", "alpha_min"]].copy()

# suma el criterio de "tocó el piso de la grilla" (recalculado, no viene guardado en `barrido`)
filas_sospechosas["toco_piso_grilla"] = [
    idx_min_en_piso(r.nivel, r.modo, r.k) for r in filas_sospechosas.itertuples()
]
filas_sospechosas = filas_sospechosas[filas_sospechosas["toco_piso_grilla"] | (filas_sospechosas["n_activos"] >= 4)]
print(f"{len(filas_sospechosas)} combinaciones sospechosas")
filas_sospechosas

[provincial] excluye 1 fila(s) por NaN: ['provincial_2001_2003']
[provincial] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[provincial] excluye 1 fila(s) por NaN: ['provincial_2001_2003']
[provincial] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[provincial] excluye 3 fila(s) por NaN: ['provincial_2003_2005', 'provincial_2013_2015', 'provincial_2015_2017']
[provincial] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[provincial] excluye 3 fila(s) por NaN: ['provincial_2003_2005', 'provincial_2013_2015', 'provincial_2015_2017']
[provincial] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[nacional] excluye 1 fila(s) por NaN: ['nacional_2001_2003']
[nacional] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[nacional] excluye 3 fila(s) por NaN: ['nacional_2003_2005', 'nacional_2013_2015', 'nacional_2015_2017']
[nacional] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[nacional] excluye 3 fila(s) por NaN: ['nacional_2003_2005', 'nacional_2013_2015', 'nacional_2015_2017']
[nacional] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


7 combinaciones sospechosas


,nivel,modo,k,n_activos,alpha_min,toco_piso_grilla
22,provincial,nivel,7,5,1.745978,False
23,provincial,nivel,8,4,2.217387,False
24,provincial,delta,1,5,1.172897,False
25,provincial,delta,2,9,0.012931,True
39,nacional,nivel,8,12,0.020556,False
40,nacional,delta,1,7,0.231592,False
41,nacional,delta,2,8,0.052519,False


In [9]:
filas_grilla_fina = []
for r in filas_sospechosas.itertuples():
    df = _panel_k(r.nivel, k=r.k, modo=r.modo)
    cols = columnas_candidatas_k(df, universo="completo", excluir_adicional=(TARGET,))
    corr = df[cols].corr(method="pearson")
    clusters = encontrar_redundantes(corr, 0.90)
    representantes = sorted(elegir_representante(c, df=df, orden_sufijo=ORDEN_SUFIJO_K) for c in clusters)
    paneles = {r.nivel: df}
    X, y = construir_Xy_final(r.nivel, representantes, paneles, target=TARGET)
    baseline = baseline_trivial_loocv(y)

    cv_10 = lasso_loocv_manual(X, y, n_alphas=10)
    cv_50 = lasso_loocv_manual(X, y, n_alphas=50)

    for etiqueta, cv in [("n_alphas=10 (original)", cv_10), ("n_alphas=50 (grilla fina)", cv_50)]:
        idx_min = cv["mean_mse"].argmin()
        mse_min = cv["mean_mse"][idx_min]
        filas_grilla_fina.append({
            "nivel": r.nivel, "modo": r.modo, "k": r.k, "grilla": etiqueta,
            "alpha_min": cv["alpha_min"],
            "mejora_pct": 100 * (1 - mse_min / baseline) if baseline else None,
        })

comparacion_grilla = pd.DataFrame(filas_grilla_fina)
comparacion_grilla.pivot_table(index=["nivel", "modo", "k"], columns="grilla", values=["alpha_min", "mejora_pct"])

[provincial] excluye 1 fila(s) por NaN: ['provincial_2001_2003']
[provincial] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[provincial] excluye 1 fila(s) por NaN: ['provincial_2001_2003']
[provincial] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[provincial] excluye 3 fila(s) por NaN: ['provincial_2003_2005', 'provincial_2013_2015', 'provincial_2015_2017']
[provincial] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[provincial] excluye 3 fila(s) por NaN: ['provincial_2003_2005', 'provincial_2013_2015', 'provincial_2015_2017']
[provincial] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[nacional] excluye 1 fila(s) por NaN: ['nacional_2001_2003']
[nacional] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[nacional] excluye 3 fila(s) por NaN: ['nacional_2003_2005', 'nacional_2013_2015', 'nacional_2015_2017']
[nacional] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[nacional] excluye 3 fila(s) por NaN: ['nacional_2003_2005', 'nacional_2013_2015', 'nacional_2015_2017']
[nacional] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


alpha_min                                       mejora_pct                          
grilla             n_alphas=10 (original) n_alphas=50 (grilla fina) n_alphas=10 (original) n_alphas=50 (grilla fina)
nivel      modo  k                                                                                                  
nacional   delta 1               0.231592                  0.207541              65.809390                 65.813124
                 2               0.052519                  0.046334              79.170130                 79.516097
           nivel 8               0.020556                  0.012649              23.161014                 23.491664
provincial delta 1               1.172897                  1.415444              60.187176                 60.917058
                 2               0.012931                  0.014889              59.928668                 59.965033
           nivel 7               1.745978                  1.296543              20.540060                 27.706894
                 8               2.217387                  1.646606              44.354059                 53.407667

### Paso 8 -- `verificar_saturacion` en los mismos casos sospechosos

`verificar_saturacion` (ya en `lasso.py`, nunca corrida sobre `panel_k`) repite el CV con
`factor_extension` en `[0.5, 2.0]` -- si `alpha_min`/`mse_min` se estabilizan al mover el
techo de la grilla (no solo su resolución, como en el Paso 7), el mínimo es real y no un límite
artificial de rango.

In [10]:
from ml_models.lasso import verificar_saturacion

filas_saturacion = []
for r in filas_sospechosas.itertuples():
    df = _panel_k(r.nivel, k=r.k, modo=r.modo)
    cols = columnas_candidatas_k(df, universo="completo", excluir_adicional=(TARGET,))
    corr = df[cols].corr(method="pearson")
    clusters = encontrar_redundantes(corr, 0.90)
    representantes = sorted(elegir_representante(c, df=df, orden_sufijo=ORDEN_SUFIJO_K) for c in clusters)
    paneles = {r.nivel: df}
    X, y = construir_Xy_final(r.nivel, representantes, paneles, target=TARGET)

    resultado = verificar_saturacion(X, y, factores=[0.5, 2.0])
    resultado.insert(0, "nivel", r.nivel)
    resultado.insert(1, "modo", r.modo)
    resultado.insert(2, "k", r.k)
    filas_saturacion.append(resultado)

saturacion = pd.concat(filas_saturacion, ignore_index=True)
saturacion

[provincial] excluye 1 fila(s) por NaN: ['provincial_2001_2003']
[provincial] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[provincial] excluye 1 fila(s) por NaN: ['provincial_2001_2003']
[provincial] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[provincial] excluye 3 fila(s) por NaN: ['provincial_2003_2005', 'provincial_2013_2015', 'provincial_2015_2017']
[provincial] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[provincial] excluye 3 fila(s) por NaN: ['provincial_2003_2005', 'provincial_2013_2015', 'provincial_2015_2017']
[provincial] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[nacional] excluye 1 fila(s) por NaN: ['nacional_2001_2003']
[nacional] excluye columna(s) sin varianza: ['icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[nacional] excluye 3 fila(s) por NaN: ['nacional_2003_2005', 'nacional_2013_2015', 'nacional_2015_2017']
[nacional] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


[nacional] excluye 3 fila(s) por NaN: ['nacional_2003_2005', 'nacional_2013_2015', 'nacional_2015_2017']
[nacional] excluye columna(s) sin varianza: ['desocupacion_cobertura_parcial', 'hacinamiento_medio_cobertura_parcial', 'icc_cobertura_parcial', 'icg_cobertura_parcial', 'ipc_cobertura_parcial', 'reservas_cobertura_parcial', 'resultado_fiscal_cobertura_parcial', 'salario_real_cobertura_parcial', 'tc_oficial_cobertura_parcial']


,nivel,modo,k,factor_extension,techo,alpha_min,alpha_1se,mse_min,mse_en_techo
0,provincial,nivel,7,0.5,4.052055,1.311836,2.305565,168.546608,328.354936
1,provincial,nivel,7,2.0,16.208222,1.281429,2.252125,167.958326,232.730475
2,provincial,nivel,8,0.5,5.146098,1.666027,2.208672,108.825844,290.645991
3,provincial,nivel,8,2.0,20.584393,1.627411,2.157477,108.830159,232.730475
4,provincial,delta,1,0.5,5.864486,1.432139,3.336814,103.940663,237.674529
5,provincial,delta,1,2.0,23.457943,1.398943,3.259471,104.060609,266.091678
6,provincial,delta,2,0.5,6.465466,0.013083,4.235711,106.033645,172.136967
7,provincial,delta,2,2.0,25.861862,0.025862,6.315604,131.168878,266.091678
8,nacional,nivel,8,0.5,4.770684,0.012798,4.770684,129.856400,188.007628
9,nacional,nivel,8,2.0,19.082735,0.019083,19.082735,130.710293,169.775115


In [11]:
# estabilidad: ¿alpha_min/mse_min se mueven proporcionalmente al factor (real, esperable) o
# quedan pegados al techo en algun factor (mse_en_techo == mse_min, seria un limite de rango)?
saturacion["pegado_al_techo"] = np.isclose(saturacion["mse_min"], saturacion["mse_en_techo"])
saturacion[["nivel", "modo", "k", "factor_extension", "alpha_min", "mse_min", "mse_en_techo", "pegado_al_techo"]]

,nivel,modo,k,factor_extension,alpha_min,mse_min,mse_en_techo,pegado_al_techo
0,provincial,nivel,7,0.5,1.311836,168.546608,328.354936,False
1,provincial,nivel,7,2.0,1.281429,167.958326,232.730475,False
2,provincial,nivel,8,0.5,1.666027,108.825844,290.645991,False
3,provincial,nivel,8,2.0,1.627411,108.830159,232.730475,False
4,provincial,delta,1,0.5,1.432139,103.940663,237.674529,False
5,provincial,delta,1,2.0,1.398943,104.060609,266.091678,False
6,provincial,delta,2,0.5,0.013083,106.033645,172.136967,False
7,provincial,delta,2,2.0,0.025862,131.168878,266.091678,False
8,nacional,nivel,8,0.5,0.012798,129.856400,188.007628,False
9,nacional,nivel,8,2.0,0.019083,130.710293,169.775115,False


### Conclusión de los Pasos 6-8

Todo lo de abajo describe únicamente lo que mostraron las celdas de arriba, en esta corrida.

- **Paso 6 (`universo="sin_eph"`, K=1,2)**: confirma el mecanismo de D34 exactamente como se
  predijo -- N sube de 8→10 (modo delta) y de 10→11 (modo nivel) en los 3 niveles, ambos K.
  **Hallazgo adicional no anticipado**: `mejora_pct` no solo cambia por el N nuevo, sino que
  **colapsa** en varias de las filas más "sospechosas" del barrido original: `nacional delta
  K=2` pasa de +79,2% (`completo`) a **-5,0%** (`sin_eph`); `provincial delta K=2` de +59,9% a
  **-10,7%**; `nacional delta K=1` de +65,8% a +30,0%; `provincial delta K=1` de +60,2% a
  +29,1%. Esto sugiere que buena parte de la "mejora" que motivó marcar esas filas como
  sospechosas ya en la conclusión original **la aportaban justamente las variables EPH** sobre
  un `N` muy chico (8) -- consistente con la reserva de sobreajuste ya señalada, ahora con una
  causa más concreta.
- **Paso 7 (grilla fina, `n_alphas=10` vs. `50`)**: en las 7 combinaciones sospechosas
  (`n_activos>=4` o `idx_min` en el piso de la grilla), `alpha_min` se mueve al afinar la
  grilla (ej. `provincial nivel K=7`: 1,746→1,297; `nacional nivel K=8`: 0,0206→0,0126), y
  `mejora_pct` se mueve un poco más de lo ideal en 2 de las 7 (`provincial nivel K=7`:
  20,5%→27,7%; `K=8`: 44,4%→53,4%) pero se mantiene estable en las otras 5 (diferencias
  <1 punto). Ninguna cambia de signo ni de orden de magnitud.
- **Paso 8 (`verificar_saturacion`, factores 0,5/2,0)**: **`pegado_al_techo` da `False` en las
  14 filas** (7 combinaciones × 2 factores) -- el mínimo de MSE nunca coincide con el MSE en el
  techo de la grilla, en ningún caso. `alpha_min` se mantiene en el mismo orden de magnitud al
  mover el techo (ej. `provincial delta K=1`: 1,43→1,40; `nacional delta K=1`: 0,21→0,21), con
  dos excepciones de mayor movimiento relativo pero sin acercarse nunca al techo
  (`provincial delta K=2`: 0,013→0,026; `nacional nivel K=8`: 0,0128→0,0191).
- **Lectura conjunta de 7-8**: los `alpha_min` sospechosos **no son un artefacto de rango de
  grilla** (nunca pegados al techo, ampliar el rango no los mueve de forma cualitativa) -- son
  mínimos interiores genuinos, aunque moderadamente sensibles a la resolución en 2 de las 7
  filas. La fragilidad que ya se había señalado en la conclusión original (`N` chico,
  modelo casi sin regularizar) sigue siendo la explicación correcta -- esto descarta
  específicamente "límite artificial de la grilla" como causa alternativa, no la reemplaza.
- **`resultado_fiscal`**: auditoría (`saltos_de_nivel`) corrida aparte de este notebook, cerrada
  sin hallazgo -- las transiciones marcadas coinciden con eventos fiscales reales de Argentina
  (superávit post-2002, déficit 2009, ajuste 2023-2025), no un artefacto de vintage como IPC.
